In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="XnevIuXUrXGEoZQJdtkk")
project = rf.workspace("priyanshu-giri").project("tyre_detection-jiyu4-njzbj")
version = project.version(1)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 12.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to tyre_detection-1 in yolov8:: 100%|██████████| 3837/3837 [00:00<00:00, 10216.18it/s]


In [2]:
import os
import shutil
os.makedirs("/content/dataset/tyre", exist_ok=True)
folders = [
    "/content/tyre_detection-1/test/images",
    "/content/tyre_detection-1/train/images",
    "/content/tyre_detection-1/valid/images"
]
count = 0
for folder in folders:
    for img in os.listdir(folder):
        shutil.copy(os.path.join(folder, img), "/content/dataset/tyre")
        count += 1
print(count)

1916


In [4]:
os.makedirs("/content/dataset/nontyre", exist_ok=True)
!unzip /content/nontyre_cnn.zip -d /content/dataset/nontyre

Archive:  /content/nontyre_cnn.zip
   creating: /content/dataset/nontyre/nontyre_cnn/
  inflating: /content/dataset/nontyre/nontyre_cnn/africa-african-animal-cat-41315.jpeg  
  inflating: /content/dataset/nontyre/nontyre_cnn/africa-animal-big-brown-41176.jpeg  
  inflating: /content/dataset/nontyre/nontyre_cnn/africa-animal-big-carnivore-41178.jpeg  
  inflating: /content/dataset/nontyre/nontyre_cnn/africa-animals-zoo-tiger.jpg  
  inflating: /content/dataset/nontyre/nontyre_cnn/animal-2607__340.jpg  
  inflating: /content/dataset/nontyre/nontyre_cnn/animal-africa-wilderness-zoo.jpg  
  inflating: /content/dataset/nontyre/nontyre_cnn/animal-world-3193850__340.jpg  
  inflating: /content/dataset/nontyre/nontyre_cnn/cat001.jpg  
  inflating: /content/dataset/nontyre/nontyre_cnn/cat002.jpg  
  inflating: /content/dataset/nontyre/nontyre_cnn/cat003.jpg  
  inflating: /content/dataset/nontyre/nontyre_cnn/cat004.jpg  
  inflating: /content/dataset/nontyre/nontyre_cnn/cat005.jpg  
  inflating

In [7]:
src = "/content/dataset/nontyre/nontyre_cnn"
dst = "/content/dataset/nontyre"
for img in os.listdir(src):
    shutil.move(os.path.join(src, img), dst)

In [8]:
os.rmdir("/content/dataset/nontyre/nontyre_cnn")

In [9]:
!pip install split-folders

In [10]:
import splitfolders
splitfolders.ratio(
    "/content/dataset",
    output="/content/dataset_split",
    seed=42,
    ratio=(0.8, 0.1, 0.1)
)

Copying files: 3770 files [00:05, 733.06 files/s]


In [11]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
img_size = (224, 224)
batch_size = 32
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
)
test_datagen = ImageDataGenerator(
    rescale=1./255,
)
train_generator = train_datagen.flow_from_directory(
    "/content/dataset_split/train",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary",
)
test_generator = test_datagen.flow_from_directory(
    "/content/dataset_split/test",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary",
    shuffle = False
)
valid_generator = test_datagen.flow_from_directory(
    "/content/dataset_split/val",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary",
)


Found 3014 images belonging to 2 classes.
Found 379 images belonging to 2 classes.
Found 376 images belonging to 2 classes.


In [12]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [13]:
base_model.summary()

Model: "mobilenetv2_1.00_224"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 2,223,872 (8.48 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [14]:
for layer in base_model.layers:
    layer.trainable = False

In [15]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation="relu")(x)
predictions = Dense(1, activation="sigmoid")(x)
model = Model(inputs=base_model.input, outputs=predictions)

In [16]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 3,570,753 (13.62 MB)

 Trainable params: 1,312,769 (5.01 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [18]:
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name ='recall')
        ]
)

In [19]:
result = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=50
)

Epoch 1/50
95/95 ━━━━━━━━━━━━━━━━━━━━ 223s 2s/step - accuracy: 0.9741 - loss: 0.0828 - precision: 0.9733 - recall: 0.9758 - val_accuracy: 0.9973 - val_loss: 0.0049 - val_precision: 0.9948 - val_recall: 1.0000
Epoch 2/50
95/95 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.9997 - loss: 0.0026 - precision: 1.0000 - recall: 0.9993 - val_accuracy: 0.9973 - val_loss: 0.0066 - val_precision: 0.9948 - val_recall: 1.0000
Epoch 3/50
95/95 ━━━━━━━━━━━━━━━━━━━━ 205s 2s/step - accuracy: 1.0000 - loss: 6.4192e-04 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 0.9973 - val_loss: 0.0050 - val_precision: 0.9948 - val_recall: 1.0000
Epoch 4/50
95/95 ━━━━━━━━━━━━━━━━━━━━ 210s 2s/step - accuracy: 1.0000 - loss: 5.2745e-04 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 0.9973 - val_loss: 0.0039 - val_precision: 0.9948 - val_recall: 1.0000
Epoch 5/50
95/95 ━━━━━━━━━━━━━━━━━━━━ 251s 2s/step - accuracy: 1.0000 - loss: 9.7879e-05 - precision: 1.0000 - recall: 1.0000 - val_accuracy: 0.9973 - val_l

In [20]:
test_acc, test_loss, test_precision, test_recall = model.evaluate(test_generator)
print(test_acc, test_loss, test_precision, test_recall)

12/12 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 1.0000 - loss: 1.5892e-06 - precision: 1.0000 - recall: 1.0000
1.5891877183094039e-06 1.0 1.0 1.0


In [27]:
!unzip /tyre_test.zip

Archive:  /tyre_test.zip
   creating: tyre_test/
  inflating: tyre_test/new_163.jpg   
  inflating: tyre_test/new_164.jpg   
  inflating: tyre_test/new_165.jpg   
  inflating: tyre_test/new_166.jpg   
  inflating: tyre_test/new_167.jpg   
  inflating: tyre_test/serviceable_297.jpg  
  inflating: tyre_test/serviceable_298.jpg  
  inflating: tyre_test/serviceable_299.jpg  
  inflating: tyre_test/serviceable_300.jpg  
  inflating: tyre_test/serviceable_301.jpg  
  inflating: tyre_test/unusable_050.jpg  
  inflating: tyre_test/unusable_051.jpg  
  inflating: tyre_test/unusable_052.jpg  
  inflating: tyre_test/unusable_053.jpg  
  inflating: tyre_test/unusable_054.jpg  


In [30]:
!unzip /nontyre_test.zip

Archive:  /nontyre_test.zip
   creating: nontyre_test/
  inflating: nontyre_test/cat064.jpg  
  inflating: nontyre_test/cat065.jpg  
  inflating: nontyre_test/cat066.jpg  
  inflating: nontyre_test/cat067.jpg  
  inflating: nontyre_test/cat068.jpg  
  inflating: nontyre_test/dog074.jpg  
  inflating: nontyre_test/dog075.jpg  
  inflating: nontyre_test/dog076.jpg  
  inflating: nontyre_test/dog077.jpg  
  inflating: nontyre_test/dog078.jpg  
  inflating: nontyre_test/fox088.jpg  
  inflating: nontyre_test/fox089.jpg  
  inflating: nontyre_test/fox090.jpg  
  inflating: nontyre_test/fox091.jpg  
  inflating: nontyre_test/fox092.jpg  


In [31]:
os.makedirs("/content/img_test/tyre", exist_ok=True)
src = "/content/tyre_test"
dst = "/content/img_test/tyre"
for img in os.listdir(src):
    shutil.move(os.path.join(src, img), dst)

In [32]:
os.makedirs("/content/img_test/non_tyre", exist_ok=True)
src = "/content/nontyre_test"
dst = "/content/img_test/non_tyre"
for img in os.listdir(src):
    shutil.move(os.path.join(src, img), dst)

In [33]:
img_size = (224, 224)
batch_size = 32
custom_datagen = ImageDataGenerator(rescale=1./255)
custom_generator = custom_datagen.flow_from_directory(
    "/content/img_test",
    target_size=img_size,
    batch_size=batch_size,
    class_mode="binary",
    shuffle = False
)

Found 30 images belonging to 2 classes.


In [34]:
cust_acc, cust_loss, cust_precision, cust_recall = model.evaluate(custom_generator)
print(cust_acc, cust_loss, cust_precision, cust_recall)

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.9000 - loss: 1.6774 - precision: 1.0000 - recall: 0.8000
1.6774163246154785 0.8999999761581421 1.0 0.800000011920929


In [35]:
model.save("tyre_classifiercnn.keras")